In [1]:
from os.path import join, basename, exists, dirname, abspath
from glob import glob
import json
import subprocess
import pdal


In [4]:
dir = abspath('C:/Users/RDCRLSMC/Desktop/20241107_Camas/09_EXPORT')
laz_fps = glob(join(dir,'*.laz'))
mosaic_fp = join(dir, 'merge.laz')

for file in laz_fps:
    print(file)


C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_171411_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_172248_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_173245_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_174213_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_175128_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_180106_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_181011_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_181850_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_182312_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_20241107241107_182847_Scanner_1.laz
C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\camas_2024110724110

## Merge LAZ files 

In [3]:
pipeline = {
    "pipeline": [
        *[
            {"type": "readers.las", "filename": fp} for fp in laz_fps
        ],
        {"type": "filters.merge"},
        {"type": "writers.las", "filename": mosaic_fp}
    ]
}

# Convert the pipeline dictionary to a JSON string
pipeline_str = json.dumps(pipeline)

# Create and execute the pipeline
p = pdal.Pipeline(pipeline_str)
count = p.execute()  # Execute the pipeline

print(f"Merged {len(laz_fps)} files into '{mosaic_fp}'. Processed {count} points.")

Merged 2 files into 'C:\Users\RDCRLSMC\Desktop\20241107_Camas\09_EXPORT\delete\merge.laz'. Processed 337544037 points.


## The following code builds a pipeline of all filters used in iceroads (adapted from notebooks/pipeline.ipynb) Scroll down to walk through filters and visualize points removed one by one

In [ ]:

# good docs on types of filters used: https://pdal.io/stages/filters.html#ground-unclassified
# Reads in mosaiced las file
reader = {"type": "readers.las", "filename": mosaic_fp}

# Filters out points with 0 returns
mongo_filter = {"type": "filters.mongo",\
    "expression": {"$and": [\
        {"ReturnNumber": {"$gt": 0}},\
             {"NumberOfReturns": {"$gt": 0}} ] } }
# Filter out points far away from our dem
dem_filter = {
        "type":"filters.dem",
        "raster":dem_fp,
        "limits":"Z[25:35]"
    }
# Extended Local Minimum filter ()
elm_filter = {"type": "filters.elm"}

outlier_filter = {"type": "filters.outlier",\
    "method": "statistical",\
        "mean_k": 12,\
            "multiplier": 2.2}

smrf_classifier = {"type": "filters.smrf",\
    "ignore": "Classification[7:7], NumberOfReturns[0:0], ReturnNumber[0:0]"}

smrf_selecter = { 
        "type":"filters.range",
        "limits":"Classification[2:2]"
    }

las_writer = {"type": "writers.las",\
#     "where": "Classification[2:2]",\
        "filename":outlas}

tif_writer = {"type": "writers.gdal",\
#     "where": "Classification[2:2]",\
        "filename":outtif,
        "resolution":1.0,
        "output_type":"idw"}


In [ ]:
pipeline = [reader, mongo_filter, dem_filter, elm_filter, outlier_filter, smrf_classifier,smrf_selecter, las_writer, tif_writer]

pipeline_str = json.dumps(pipeline)

# Create and execute the pipeline
p = pdal.Pipeline(pipeline_str)
count = p.execute()  # Execute the pipeline
